In [2]:
## DB 연결

from neo4j import GraphDatabase

# URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
URI = "neo4j+s://7083bdc2.databases.neo4j.io"
AUTH = ("neo4j", "Du6u10X1_TE2Zviv8YyG5EgoN4t-m_nvZhQfw3cyato")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

In [3]:
## Create an example graph

summary = driver.execute_query("""
    CREATE (a:KC {name: $pre_name, id: 1, category: "machine learning", link: "https://en.wikipedia.org/wiki/Attention_(machine_learning)"})  
    CREATE (b:KC {name: $name, id: 2, category: "deep learning", link: "https://en.wikipedia.org/wiki/Transformer_(deep_learning)"})
    CREATE (a)-[:PREREQ {id: 1, strength: 1, reason: "Transformer는 RNN/CNN 대신 self-attention으로 토큰 간 의존성을 계산하며, 이 self-attention 메커니즘이 Attention 개념을 직접 확장한 것이다."}]->(b)
    """,
    ## a라는 변수로 KC 노드를 만들고 name 속성에 $pre_name 값을 넣음
    ## b라는 변수로 KC 노드를 만들고 name 속성에 $name 값을 넣음
    ## a와 b 사이에 PREREQ 관계를 만듦
    pre_name="Attention", name="Transformer",
    database_="neo4j",
).summary
print("Created {nodes_created} nodes in {time} ms.".format(
    nodes_created=summary.counters.nodes_created,
    time=summary.result_available_after
))

/tmp/ipykernel_3921666/3011818324.py:3: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  summary = driver.execute_query("""


Created 2 nodes in 58 ms.


In [6]:
## Query a graph

# Transformer의 선수 개념을 모두 찾아보기
records, summary, keys = driver.execute_query("""
    MATCH (a:KC)-[:PREREQ]->(b:KC {name: $name})
    RETURN a.name AS prerequisite
    """,  # a.name을 prerequisite라는 컬럼 이름으로 반환
    name="Transformer",
    database_="neo4j",
)

for record in records:
    print(f"Prerequisite: {record['prerequisite']}")

# property까지 함께 출력하기
print("\n--- With properties ---")
records, summary, keys = driver.execute_query("""
    MATCH (a:KC)-[:PREREQ]->(b:KC {name: $name})
    RETURN properties(a) AS prerequisite_props
""", name="Transformer", database_="neo4j")

for r in records:
    props = r["prerequisite_props"]   # dict 형태
    print(props)

/tmp/ipykernel_3921666/2170036688.py:4: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = driver.execute_query("""
[#9C7C]  _: <CONNECTION> error: Failed to write data to connection IPv4Address(('si-7083bdc2-4350.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))): ConnectionResetError(104, 'Connection reset by peer')
Transaction failed and will be retried in 0.9984544459981253s (Failed to write data to connection IPv4Address(('si-7083bdc2-4350.production-orch-0066.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.114.186', 7687))))


Prerequisite: Attention

--- With properties ---


/tmp/ipykernel_3921666/2170036688.py:17: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = driver.execute_query("""


{'name': 'Attention', 'link': 'https://en.wikipedia.org/wiki/Attention_(machine_learning)', 'id': 1, 'category': 'machine learning'}
